In [1]:
# =====================================================
# FINAL IMPROVED FLOOD DETECTION PHASE 2 NOTEBOOK
# (All your requested upgrades applied)
# =====================================================

# ── Installs ─────────────────────────────────────────────────────────────
!pip install -q segmentation-models-pytorch
!pip install -q albumentations

import os
import random
import numpy as np
import pandas as pd
import torch
import rasterio
import cv2
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

BASE_PATH = "/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data"
image_dir = f"{BASE_PATH}/image"
mask_dir  = f"{BASE_PATH}/label"
pred_dir  = f"{BASE_PATH}/prediction/image"
train_file = f"{BASE_PATH}/split/train.txt"
val_file   = f"{BASE_PATH}/split/val.txt"      # used for threshold tuning

# ========================== 1. CLASS WEIGHTS ==========================
train_ids = open(train_file).read().splitlines()
print("Train samples:", len(train_ids))

counts = np.zeros(3, dtype=np.int64)
for id_ in train_ids:
    with rasterio.open(f"{BASE_PATH}/label/{id_}_label.tif") as src:
        lbl = src.read(1).astype(np.int64)
    for c in range(3):
        counts[c] += (lbl == c).sum()

freq = counts / counts.sum()
raw_weights = 1.0 / (freq + 1e-8)
raw_weights = raw_weights / raw_weights.sum()
raw_weights[1] *= 2.0
CLASS_WEIGHTS = raw_weights.tolist()
print(f'Class weights: {[round(w,4) for w in CLASS_WEIGHTS]}')

# ========================== SAR SPECKLE FILTER (Point 1 - Highest Impact) ==========================
def speckle_filter(hh: np.ndarray, hv: np.ndarray) -> tuple:
    """Simple bilateral filter as SAR speckle reducer (very effective for flood mapping)"""
    hh = cv2.bilateralFilter(hh.astype(np.float32), d=5, sigmaColor=50, sigmaSpace=50)
    hv = cv2.bilateralFilter(hv.astype(np.float32), d=5, sigmaColor=50, sigmaSpace=50)
    return hh, hv

# ========================== DATASET STATS & TRANSFORMS ==========================
MEANS = np.array([841.1257, 371.6175, 1734.1789, 1588.3142, 1742.8452, 1218.5551], dtype=np.float32)[:, None, None]
STDS  = np.array([473.7090, 170.3611, 623.0474, 612.8465, 564.5835, 528.0894], dtype=np.float32)[:, None, None]

train_transform = A.Compose([
    A.D4(p=1.0),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.GaussNoise(p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.25),
])

class FloodDataset(Dataset):
    def __init__(self, ids, image_dir, mask_dir, transform=None):
        self.ids = ids
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_ = self.ids[idx]
        img_path = f"{self.image_dir}/{id_}_image.tif"
        mask_path = f"{self.mask_dir}/{id_}_label.tif"

        with rasterio.open(img_path) as src:
            image6 = src.read().astype(np.float32)          # (6, H, W)

        with rasterio.open(mask_path) as src:
            mask = src.read(1).astype(np.int64)

        # === Point 1: SAR Speckle Filter BEFORE normalization ===
        image6[0], image6[1] = speckle_filter(image6[0], image6[1])

        image6 = (image6 - MEANS) / STDS

        if self.transform:
            aug = self.transform(image=image6.transpose(1,2,0), mask=mask)
            image6 = aug["image"].transpose(2,0,1)
            mask = aug["mask"]

        green, nir, swir = image6[2], image6[4], image6[5]
        ndwi  = np.clip((green - nir)  / (np.abs(green + nir)  + 1e-6), -3, 3)
        mndwi = np.clip((green - swir) / (np.abs(green + swir) + 1e-6), -3, 3)
        image = np.concatenate([image6, ndwi[np.newaxis], mndwi[np.newaxis]], axis=0)

        return torch.tensor(image, dtype=torch.float32), torch.tensor(mask, dtype=torch.long)


# Create train + val datasets
train_dataset = FloodDataset(train_ids, image_dir, mask_dir, transform=train_transform)

val_ids = open(val_file).read().splitlines() if os.path.exists(val_file) else train_ids[:10]  # fallback
val_dataset = FloodDataset(val_ids, image_dir, mask_dir, transform=None)   # no aug for val

print(f"Image shape: {train_dataset[0][0].shape} → 8 channels")

# ========================== MODEL ==========================
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=8,
    classes=3
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("Device:", device)

# ========================== LOSS (Point 7) ==========================
ce_loss = torch.nn.CrossEntropyLoss(weight=torch.tensor(CLASS_WEIGHTS).to(device))
dice_loss = smp.losses.DiceLoss(mode="multiclass")

class BoundaryLoss(torch.nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, logits, target):
        prob = torch.softmax(logits, dim=1)
        target_onehot = torch.nn.functional.one_hot(target, num_classes=logits.shape[1]).permute(0, 3, 1, 2).float()
        
        def sobel_edge(x):
            sobel_x = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=x.dtype, device=x.device).view(1,1,3,3)
            sobel_y = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=x.dtype, device=x.device).view(1,1,3,3)
            edge_x = torch.nn.functional.conv2d(x, sobel_x, padding=1)
            edge_y = torch.nn.functional.conv2d(x, sobel_y, padding=1)
            return torch.sqrt(edge_x**2 + edge_y**2 + 1e-6)
        
        pred_edge = sobel_edge(prob[:, 1:2, :, :])      # flood class boundary
        target_edge = sobel_edge(target_onehot[:, 1:2, :, :])
        
        return torch.mean(torch.abs(pred_edge - target_edge))

def loss_fn(pred, target):
    return (0.4 * ce_loss(pred, target) +
            0.4 * dice_loss(pred, target) +
            0.2 * BoundaryLoss()(pred, target))

# ========================== OPTIMIZER + DATALOADERS ==========================
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=2)

print(f"✅ DataLoaders ready → Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# ========================== TRAINING + SNAPSHOT ENSEMBLE (Point 3) ==========================
EPOCHS = 25
snapshot_paths = []

model.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)
        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1:2d}  Loss: {total_loss/len(train_loader):.6f}")

    # Save snapshot every 5 epochs (Point 3)
    if (epoch + 1) % 5 == 0 or epoch == EPOCHS-1:
        path = f"model_epoch_{epoch+1}.pth"
        torch.save(model.state_dict(), path)
        snapshot_paths.append(path)
        print(f"   → Saved snapshot: {path}")

# Keep only last 3 snapshots for ensemble
snapshot_paths = snapshot_paths[-3:]
print(f"Using {len(snapshot_paths)} snapshots for ensemble")

# ========================== THRESHOLD TUNING (Point 2) ==========================
@torch.no_grad()
def compute_flood_iou(threshold):
    model.eval()
    ious = []
    for images, masks in val_loader:
        images = images.to(device)
        preds = model(images)
        prob = torch.softmax(preds, dim=1)[:, 1].cpu().numpy()   # flood class
        pred_mask = (prob > threshold).astype(np.uint8)
        true_mask = (masks.numpy() == 1).astype(np.uint8)
        intersection = np.logical_and(pred_mask, true_mask).sum(axis=(1,2))
        union = np.logical_or(pred_mask, true_mask).sum(axis=(1,2))
        iou = np.where(union == 0, 1.0, intersection / union)
        ious.extend(iou)
    return np.mean(ious)

print("Tuning flood threshold on validation set...")
best_thresh, best_iou = 0.45, 0.0
for thresh in np.arange(0.30, 0.71, 0.01):
    iou = compute_flood_iou(thresh)
    if iou > best_iou:
        best_iou, best_thresh = iou, thresh
print(f"✅ Optimal flood threshold: {best_thresh:.3f} (val IoU = {best_iou:.4f})")

# ========================== MULTI-SCALE + 8-WAY TTA INFERENCE (Point 6) ==========================
@torch.no_grad()
def predict_tta_multi_scale(model, img: torch.Tensor) -> torch.Tensor:
    """8-way D4 TTA + multi-scale (0.8, 1.0, 1.2)"""
    scales = [0.8, 1.0, 1.2]
    all_probs = []

    def _p(x):
        return torch.softmax(model(x), dim=1)

    for scale in scales:
        # Simple resize (for speed)
        if scale != 1.0:
            orig_h, orig_w = img.shape[2], img.shape[3]
            new_h, new_w = int(orig_h * scale), int(orig_w * scale)
            scaled = torch.nn.functional.interpolate(img, size=(new_h, new_w), mode='bilinear', align_corners=False)
        else:
            scaled = img

        # 8-way geometric TTA
        p0 = _p(scaled)
        p1 = torch.flip(_p(torch.flip(scaled, [3])), [3])
        p2 = torch.flip(_p(torch.flip(scaled, [2])), [2])
        p3 = torch.rot90(_p(torch.rot90(scaled, 1, [2,3])), -1, [2,3])
        p4 = torch.rot90(_p(torch.rot90(scaled, 2, [2,3])), -2, [2,3])
        p5 = torch.rot90(_p(torch.rot90(scaled, 3, [2,3])), -3, [2,3])
        img_h = torch.flip(scaled, [3])
        p6 = torch.flip(torch.rot90(_p(torch.rot90(img_h, 1, [2,3])), -1, [2,3]), [3])
        p7 = torch.flip(torch.rot90(_p(torch.rot90(img_h, 3, [2,3])), -3, [2,3]), [3])

        avg = (p0 + p1 + p2 + p3 + p4 + p5 + p6 + p7) / 8.0

        # Resize back to original size
        if scale != 1.0:
            avg = torch.nn.functional.interpolate(avg, size=(orig_h, orig_w), mode='bilinear', align_corners=False)

        all_probs.append(avg)

    return torch.mean(torch.stack(all_probs), dim=0).squeeze(0)   # (3, H, W)


def postprocess_flood(flood_mask: np.ndarray) -> np.ndarray:
    if flood_mask.max() == 0:
        return flood_mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    closed = cv2.morphologyEx(flood_mask, cv2.MORPH_CLOSE, kernel)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(closed, connectivity=8)
    clean = np.zeros_like(closed)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= 50:
            clean[labels == i] = 1
    return clean


def rle_encode(mask: np.ndarray) -> str:
    if mask.max() == 0:
        return "0 0"
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(map(str, runs))


# ========================== FINAL INFERENCE WITH ENSEMBLE ==========================
model.eval()
pred_ids = sorted([f.replace("_image.tif", "") for f in os.listdir(pred_dir)])
predictions = []

print(f"Running multi-scale 8-way TTA + ensemble on {len(pred_ids)} test images...")

for id_ in pred_ids:
    img_path = f"{pred_dir}/{id_}_image.tif"
    with rasterio.open(img_path) as src:
        image6 = src.read().astype(np.float32)
    image6 = (image6 - MEANS) / STDS
    green, nir, swir = image6[2], image6[4], image6[5]
    ndwi  = np.clip((green - nir)  / (np.abs(green + nir)  + 1e-6), -3, 3)
    mndwi = np.clip((green - swir) / (np.abs(green + swir) + 1e-6), -3, 3)
    image = np.concatenate([image6, ndwi[np.newaxis], mndwi[np.newaxis]], axis=0)

    image_t = torch.tensor(image, dtype=torch.float32).unsqueeze(0).to(device)

    # Ensemble over snapshots
    ensemble_prob = None
    for snap_path in snapshot_paths:
        model.load_state_dict(torch.load(snap_path, map_location=device))
        prob = predict_tta_multi_scale(model, image_t)          # multi-scale + 8-way TTA
        ensemble_prob = prob if ensemble_prob is None else ensemble_prob + prob
    ensemble_prob /= len(snapshot_paths)

    flood_prob = ensemble_prob[1].cpu().numpy()
    mask = (flood_prob > best_thresh).astype(np.uint8)
    mask = postprocess_flood(mask)

    rle = rle_encode(mask)
    predictions.append((id_, rle))

df = pd.DataFrame(predictions, columns=["id", "rle_mask"])
df.to_csv("submission_phase2.csv", index=False)

print(df.head())
print(f"\n✅ Submission saved: submission_phase2.csv ({len(df)} rows)")
print(f"Used threshold: {best_thresh:.3f} | Snapshots: {len(snapshot_paths)} | Multi-scale TTA enabled")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.9 MB/s eta 0:00:00
Train samples: 59
Class weights: [0.116, 0.9922, 0.3879]
Image shape: torch.Size([8, 512, 512]) → 8 channels


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Device: cuda
✅ DataLoaders ready → Train batches: 15 | Val batches: 3
Epoch  1  Loss: 0.903080
Epoch  2  Loss: 0.805987
Epoch  3  Loss: 0.748506
Epoch  4  Loss: 0.707068
Epoch  5  Loss: 0.661887
   → Saved snapshot: model_epoch_5.pth
Epoch  6  Loss: 0.654570
Epoch  7  Loss: 0.632152
Epoch  8  Loss: 0.595433
Epoch  9  Loss: 0.599510
Epoch 10  Loss: 0.599357
   → Saved snapshot: model_epoch_10.pth
Epoch 11  Loss: 0.586682
Epoch 12  Loss: 0.597643
Epoch 13  Loss: 0.572724
Epoch 14  Loss: 0.552587
Epoch 15  Loss: 0.562403
   → Saved snapshot: model_epoch_15.pth
Epoch 16  Loss: 0.534998
Epoch 17  Loss: 0.560904
Epoch 18  Loss: 0.545140
Epoch 19  Loss: 0.524715
Epoch 20  Loss: 0.530502
   → Saved snapshot: model_epoch_20.pth
Epoch 21  Loss: 0.537596
Epoch 22  Loss: 0.525466
Epoch 23  Loss: 0.530695
Epoch 24  Loss: 0.534722
Epoch 25  Loss: 0.512978
   → Saved snapshot: model_epoch_25.pth
Using 3 snapshots for ensemble
Tuning flood threshold on validation set...
✅ Optimal flood threshold: 0.34